# HanBayes 交互式演示

本笔记本演示 **HanBayes**（可解释中文情感贝叶斯模型族）的完整流程：

1. **数据准备**：加载 ChnSentiCorp 数据并执行两阶段清洗；
2. **一行训练四个模型**：StandardNB / FWNB / DFWNB-v2 / SDFWNB 共享单次扫描的稀疏矩阵；
3. **评估**：Accuracy / Macro-F1 / AUC；
4. **精确解释**：把任意预测拆解到特征证据与依赖修正。

> 首次使用请先在仓库根目录运行 `hanbayes download` 下载数据集，
> 或直接运行下方单元格。

In [ ]:
import sys
from pathlib import Path

# 支持从源码目录直接运行（无需 pip install）
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from hanbayes.configs import load_frozen_config
from hanbayes.data.download import download_dataset
from hanbayes.data.io import read_dataset_file, standardize_dataset_columns
from hanbayes.data.cleaning import clean_and_prepare

CONFIG = load_frozen_config()
print("冻结配置加载成功，alpha =", CONFIG["naive_bayes"]["alpha"])

## 1. 数据准备

下载数据集（带 SHA-256 校验），并执行论文中的两阶段清洗协议：

- **第一阶段（原始文本级）**：删除集内标签冲突、集内重复、跨集合重叠（test > dev > train 优先级）；
- **第二阶段（模型文本级）**：对规范化后的 `model_text` 重复上述检查，防止归一化产生的新碰撞与泄漏。

In [ ]:
DATA_DIR = REPO_ROOT / "data" / "raw"
download_dataset(DATA_DIR)

raw = {}
for split in ("train", "dev", "test"):
    raw[split] = standardize_dataset_columns(
        read_dataset_file(DATA_DIR / f"{split}.tsv"), split
    )

prepared = clean_and_prepare(raw["train"], raw["dev"], raw["test"])
ready = prepared["ready"]

print(prepared["model_summary"].to_string(index=False))
print()
for split in ("train", "dev", "test"):
    print(f"{split}: {len(raw[split])} -> {len(ready[split])}")

## 2. 一行训练四个模型

`ChineseSentimentAnalyzer` 把语料**只扫描一次**，产出单个稀疏计数矩阵，
四个模型共享；训练 + 评估全流程向量化，秒级完成。

In [ ]:
from hanbayes import ChineseSentimentAnalyzer

analyzer = ChineseSentimentAnalyzer.from_config()
analyzer.fit(
    ready["train"]["model_text"].tolist(),
    ready["train"]["label"].astype(int).to_numpy(),
)
report = analyzer.evaluate(
    ready["dev"]["model_text"].tolist(),
    ready["dev"]["label"].astype(int).to_numpy(),
)
report.frame

## 3. 推理与概率

训练完成后即可对新文本预测情感（1 = 正面，0 = 负面）。

In [ ]:
probe_texts = [
    "房间干净整洁，服务人员很热情，下次还会再来",
    "设施老旧，空调噪音很大，早餐品类少，不值这个价格",
]

preds = analyzer.predict(probe_texts)
proba = analyzer.predict_proba(probe_texts)
for text, label, p in zip(probe_texts, preds, proba):
    sentiment = "正面" if label == 1 else "负面"
    print(f"[{sentiment}] 正面概率={p:.4f} | {text}")

## 4. 精确解释（HanBayes 的招牌能力）

模型族对特征证据是**线性**的，因此每个预测都能精确分解——无 LIME、无近似：

In [ ]:
import json

expl = analyzer.explainer.explain_prediction(
    "房间干净服务好但隔音差", model_name="SDFWNB", top_n=8
)
print("预测类别:", expl["predicted_class"],
      "概率:", [round(p, 4) for p in expl["probabilities"]])
print()
print("== 特征证据 (top)")
for item in expl["feature_evidence"][:5]:
    print(f"  {item['feature']}: 负面证据 {item['evidence_negative']:+.3f} | "
          f"正面证据 {item['evidence_positive']:+.3f}")
print()
print("== 依赖修正 (top)")
for item in expl["dependency_effects"][:5]:
    print(f"  {item['trigram']}: 负面 {item['effect_negative']:+.3f} | "
          f"正面 {item['effect_positive']:+.3f}")

### 全局解释表

In [ ]:
print("== 判别力最强的特征（按互信息）==")
display(analyzer.explainer.top_features(top_n=10).head(10))

print("== 最强的依赖三连字（SDFWNB 修正对）==")
analyzer.explainer.top_dependencies(top_n=10)

## 5. 复现论文最终数字

最终测试协议将 train+dev 合并为最终训练集，在 test 上一次性评估。
命令行一键复现：

```bash
hanbayes final-test
```

预期输出（与论文冻结数字一致）：

| 模型 | Accuracy | Macro-F1 | AUC |
|---|---|---|---|
| 标准NB | 0.7793 | 0.7786 | 0.8505 |
| FWNB | 0.8022 | 0.8009 | 0.8770 |
| DFWNB-v2 | 0.8048 | 0.8033 | 0.8822 |
| SDFWNB | **0.8073** | **0.8065** | **0.8867** |

In [ ]:
# frozen pipeline for the final-test protocol
from hanbayes.pipeline import run_frozen_pipeline

In [ ]:
import pandas as pd

merged = pd.concat([ready["train"], ready["dev"]], ignore_index=True)
final_output = run_frozen_pipeline(
    training_texts=merged["model_text"].tolist(),
    training_labels=merged["label"].astype(int).to_numpy(),
    evaluation_texts=ready["test"]["model_text"].tolist(),
    evaluation_labels=ready["test"]["label"].astype(int).to_numpy(),
)
final_output["results"]